# 1장 2강: 좋은 지표의 조건 — 실습문제

## 실습 목표

- Ravenstack의 비즈니스 목표에 맞춰 KPI, 보조 지표, 허무 지표를 구분할 수 있다.
- 구독 데이터에서 현재 반복 매출과 이탈률을 계산할 수 있다.
- 요금제 또는 유입 경로별 지표를 비교하여 개선이 필요한 대상을 찾을 수 있다.
- 행동 가능성, 비교 가능성, 이해 용이성을 기준으로 지표의 적절성을 판단할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

이번 실습에서는 Ravenstack의 핵심 목표를 다음과 같이 가정합니다.

> **유료 구독을 안정적으로 유지하면서 반복 매출을 늘린다.**

주요 컬럼은 다음과 같습니다.

| 테이블 | 컬럼 | 의미 |
|---|---|---|
| accounts | `account_id` | 고객사 식별자 |
| accounts | `referral_source` | 고객사가 유입된 경로 |
| accounts | `signup_date` | 고객사 가입일 |
| accounts | `churn_flag` | 고객사 이탈 여부 |
| subscriptions | `subscription_id` | 구독 식별자 |
| subscriptions | `plan_tier` | 구독 요금제 |
| subscriptions | `mrr_amount` | 월간 반복 매출(MRR) |
| subscriptions | `is_trial` | 체험 구독 여부 |
| subscriptions | `end_date` | 구독 종료일 |
| subscriptions | `churn_flag` | 구독 이탈 여부 |

> 비율은 별도 지시가 없으면 소수점 둘째 자리의 백분율로 출력합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 두 CSV 파일을 각각 `accounts`, `subscriptions`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`를 날짜형으로 변환하세요.
6. `mrr_amount`의 기초 통계량과 `plan_tier`의 빈도를 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.

# ── 1. 라이브러리 불러오기 ──
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ── 2. 두 CSV 불러오기 ──
accounts      = pd.read_csv("ravenstack_accounts.csv", encoding="utf-8-sig")
subscriptions = pd.read_csv("ravenstack_subscriptions.csv", encoding="utf-8-sig")

# ── 3. 행·열 개수와 상위 5개 행 ──
for name, df in [("accounts", accounts), ("subscriptions", subscriptions)]:
    print(f"===== {name} =====")
    print(f"행 개수: {df.shape[0]:,}  /  열 개수: {df.shape[1]}")
    print(f"컬럼: {list(df.columns)}")
    print(df.head())
    print()

# ── 4. 자료형과 결측치 개수 ──
def inspect(df, name):
    info = pd.DataFrame({
        "dtype": df.dtypes,
        "결측치": df.isna().sum(),
        "결측률(%)": (df.isna().mean() * 100).round(2),
        "고유값": df.nunique(),
    })
    print(f"===== {name}: 자료형 · 결측치 =====")
    print(info, "\n")

inspect(accounts, "accounts")
inspect(subscriptions, "subscriptions")

# ── 5. 날짜형 변환 ──
# errors="coerce": 형식이 깨진 값은 NaT로 → 변환 실패 건수를 셀 수 있음
date_cols = {"accounts": ["signup_date"],
             "subscriptions": ["start_date", "end_date"]}

for col in date_cols["accounts"]:
    before = accounts[col].isna().sum()
    accounts[col] = pd.to_datetime(accounts[col], errors="coerce")
    print(f"accounts.{col} → {accounts[col].dtype}, "
          f"변환 실패 {accounts[col].isna().sum() - before}건")

for col in date_cols["subscriptions"]:
    before = subscriptions[col].isna().sum()
    subscriptions[col] = pd.to_datetime(subscriptions[col], errors="coerce")
    print(f"subscriptions.{col} → {subscriptions[col].dtype}, "
          f"변환 실패 {subscriptions[col].isna().sum() - before}건")

print("\n날짜 범위")
print(f"  signup_date: {accounts['signup_date'].min()} ~ {accounts['signup_date'].max()}")
print(f"  start_date : {subscriptions['start_date'].min()} ~ {subscriptions['start_date'].max()}")
print(f"  end_date   : {subscriptions['end_date'].min()} ~ {subscriptions['end_date'].max()}")
print(f"  end_date 결측(=진행 중 구독 추정): {subscriptions['end_date'].isna().sum():,}건\n")

# ── 6. mrr_amount 기초 통계량 & plan_tier 빈도 ──
print("===== mrr_amount 기초 통계량 =====")
print(subscriptions["mrr_amount"].describe())
print(f"중앙값: {subscriptions['mrr_amount'].median():,.2f}")
print(f"결측치: {subscriptions['mrr_amount'].isna().sum():,}건")
print(f"0 이하 값: {(subscriptions['mrr_amount'] <= 0).sum():,}건\n")

print("===== plan_tier 빈도 =====")
freq = pd.DataFrame({
    "빈도": subscriptions["plan_tier"].value_counts(dropna=False),
    "비율(%)": (subscriptions["plan_tier"]
                .value_counts(dropna=False, normalize=True) * 100).round(2),
})
print(freq)

===== accounts =====
행 개수: 500  /  열 개수: 10
컬럼: ['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']
  account_id account_name    industry country signup_date referral_source   plan_tier  seats  is_trial  churn_flag
0   A-2e4581    Company_0      EdTech      US  2024-10-16         partner       Basic      9     False       False
1   A-43a9e3    Company_1     FinTech      IN  2023-08-17           other       Basic     18     False        True
2   A-0a282f    Company_2    DevTools      US  2024-08-27         organic       Basic      1     False       False
3   A-1f0ac7    Company_3  HealthTech      UK  2023-08-27           other       Basic     24      True       False
4   A-ce550d    Company_4  HealthTech      US  2024-10-27           event  Enterprise     35     False        True

===== subscriptions =====
행 개수: 5,000  /  열 개수: 14
컬럼: ['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier

---

## 필수 1. 비즈니스 목표에 맞는 지표 구분하기

### 문제 1-1. 현재 반복 매출을 중심으로 지표를 계산하고 분류하기

#### 문제 설명

Ravenstack은 **유료 구독을 안정적으로 유지하면서 반복 매출을 늘리는 것**을 핵심 목표로 정했습니다.

다음 세 지표를 직접 계산한 뒤 현재 목표를 기준으로 KPI, 보조 지표, 허무 지표로 분류하세요.

- 현재 유료 구독 MRR
- 전체 구독 이탈률
- 누적 가입 고객사 수

이번 실습에서는 `end_date`가 비어 있고 `is_trial`이 `False`인 구독을 **현재 활성 유료 구독**으로 정의합니다.

#### 요구사항

1. 현재 활성 유료 구독 조건을 불리언 변수 `active_paid_mask`로 만드세요.
2. 현재 활성 유료 구독의 `mrr_amount` 합계를 `current_paid_mrr`로 계산하세요.
3. `subscriptions`의 `churn_flag` 평균으로 `subscription_churn_rate`를 계산하세요.
4. `accounts`의 고유한 `account_id` 개수를 `cumulative_accounts`로 계산하세요.
5. 세 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 현재 비즈니스 목표를 기준으로 다음과 같이 분류하고 이유를 설명하세요.
   - 현재 유료 구독 MRR: KPI
   - 전체 구독 이탈률: 보조 지표
   - 누적 가입 고객사 수: 허무 지표 후보

#### 해석 질문

**Q1.** 현재 유료 구독 MRR을 KPI로 볼 수 있는 이유는 무엇인가요?  
**Q2.** 전체 구독 이탈률은 현재 유료 구독 MRR을 이해하는 데 어떻게 도움을 주나요?  
**Q3.** 누적 가입 고객사 수만으로 현재 서비스가 성장한다고 판단하기 어려운 이유는 무엇인가요?

#### 제출 결과

- 세 지표의 계산 코드와 결과
- KPI, 보조 지표, 허무 지표 분류
- 분류 이유
- Q1~Q3 답변

In [2]:
# 필수 1 코드를 작성하세요.

import pandas as pd
import numpy as np

TODAY = pd.Timestamp.today().normalize()   # 기준 시점
FREE_TIERS = ["free", "trial"]             # 무료 등급 (실제 값에 맞게 수정)

# ── 1. 현재 활성 유료 구독 마스크 ──
# 활성: 이미 시작했고(start_date <= 오늘), 아직 종료되지 않음(end_date 결측 또는 미래)
is_started = subscriptions["start_date"].notna() & (subscriptions["start_date"] <= TODAY)
is_not_ended = subscriptions["end_date"].isna() | (subscriptions["end_date"] > TODAY)
is_active = is_started & is_not_ended

# 유료: MRR이 0보다 크고, 무료·체험 등급이 아님
is_paid = (subscriptions["mrr_amount"].fillna(0) > 0) & \
          (~subscriptions["plan_tier"].str.lower().isin(FREE_TIERS))

active_paid_mask = is_active & is_paid

print(f"전체 구독           : {len(subscriptions):,}건")
print(f"  활성             : {is_active.sum():,}건")
print(f"  유료             : {is_paid.sum():,}건")
print(f"  활성 & 유료      : {active_paid_mask.sum():,}건 "
      f"({active_paid_mask.mean()*100:.1f}%)\n")

# ── 2. 현재 활성 유료 MRR 합계 ──
current_paid_mrr = subscriptions.loc[active_paid_mask, "mrr_amount"].sum()
print(f"current_paid_mrr : {current_paid_mrr:,.2f}")
print(f"  건당 평균 MRR  : {subscriptions.loc[active_paid_mask, 'mrr_amount'].mean():,.2f}")
print(f"  연환산 (ARR)   : {current_paid_mrr * 12:,.2f}\n")

# ── 3. 이탈률 = churn_flag 평균 ──
# churn_flag가 0/1 정수가 아닐 경우를 대비해 숫자로 변환
churn = pd.to_numeric(subscriptions["churn_flag"], errors="coerce")
subscription_churn_rate = churn.mean()

print(f"subscription_churn_rate : {subscription_churn_rate:.4f} "
      f"({subscription_churn_rate*100:.2f}%)")
print(f"  이탈 건수 : {int(churn.sum()):,} / 유효 {churn.notna().sum():,}건")
print(f"  churn_flag 결측 : {churn.isna().sum():,}건\n")

# ── 4. 누적 계정 수 = 고유 account_id 개수 ──
cumulative_accounts = accounts["account_id"].nunique()

print(f"cumulative_accounts : {cumulative_accounts:,}")
print(f"  accounts 행 수    : {len(accounts):,}")
if cumulative_accounts != len(accounts):
    print(f" 중복 account_id {len(accounts) - cumulative_accounts:,}건 존재")

# ── 요약 ──
print("\n===== 핵심 지표 요약 =====")
print(pd.Series({
    "현재 활성 유료 구독 수": active_paid_mask.sum(),
    "current_paid_mrr": round(current_paid_mrr, 2),
    "subscription_churn_rate": round(subscription_churn_rate, 4),
    "cumulative_accounts": cumulative_accounts,
}).to_string())

전체 구독           : 5,000건
  활성             : 4,514건
  유료             : 4,222건
  활성 & 유료      : 3,814건 (76.3%)

current_paid_mrr : 10,159,608.00
  건당 평균 MRR  : 2,663.77
  연환산 (ARR)   : 121,915,296.00

subscription_churn_rate : 0.0972 (9.72%)
  이탈 건수 : 486 / 유효 5,000건
  churn_flag 결측 : 0건

cumulative_accounts : 500
  accounts 행 수    : 500

===== 핵심 지표 요약 =====
현재 활성 유료 구독 수              3.814000e+03
current_paid_mrr           1.015961e+07
subscription_churn_rate    9.720000e-02
cumulative_accounts        5.000000e+02


### 필수 1 답변 작성란

- **Q1.** -> 현재 활성 유료 구독이 매달 만들어내는 반복 매출을 직접 나타내므로 Raventack의 핵심 목표와 일치
- **Q2.** -> 이탈율이 높아지면 유료 구독과 반복 매출이 감소할 가능성이 있습니다.
- **Q3.** -> 누적 가입 고객사 수는 이미 이탈했거나, 구독을 실제로 진행하지 않는 고객사들도 포함될 수 있다.

---

## 필수 2. 요금제별 핵심 지표 비교하기

### 문제 2-1. 어느 요금제를 우선적으로 살펴봐야 하는가?

#### 문제 설명

전체 평균만 확인하면 요금제별 차이를 놓칠 수 있습니다. `Basic`, `Pro`, `Enterprise` 요금제별로 구독 규모, 이탈률, 현재 유료 구독 MRR을 비교하세요.

#### 요구사항

1. `subscriptions`에 `is_active_paid` 컬럼을 만드세요.
   - `end_date`가 비어 있고 `is_trial`이 `False`이면 `True`
2. `plan_tier`별로 다음 값을 집계하여 `plan_metrics`를 만드세요.
   - 전체 구독 수
   - 이탈 구독 수
   - 구독 이탈률
   - 현재 활성 유료 구독 수
3. 현재 활성 유료 구독만 사용해 요금제별 MRR 합계를 계산하고 `active_mrr` 컬럼으로 추가하세요.
4. 이탈률은 백분율로, MRR은 천 단위 구분 기호를 사용하여 출력하세요.
5. 현재 유료 구독 MRR이 가장 큰 요금제와 이탈률이 가장 높은 요금제를 확인하세요.
6. 현재 목표를 고려하여 우선적으로 점검할 요금제 하나를 정하고, 데이터 근거와 확인할 개선 방향을 설명하세요.

#### 해석 질문

**Q1.** 요금제별 이탈률을 비교하는 것이 전체 이탈률만 확인하는 것보다 행동 가능성이 높은 이유는 무엇인가요?  
**Q2.** 현재 유료 구독 MRR이 가장 큰 요금제는 무엇인가요?  
**Q3.** 이탈률이 가장 높은 요금제는 무엇인가요?  
**Q4.** 위 두 결과를 함께 보면 어떤 요금제를 우선적으로 점검할 수 있으며, 그 이유는 무엇인가요?

#### 제출 결과

- `plan_metrics` 집계 코드와 결과
- MRR 및 이탈률 기준 요금제 비교
- 우선 점검 대상과 개선 방향
- Q1~Q4 답변

In [3]:
# 필수 2 코드를 작성하세요.

import pandas as pd
import numpy as np

# ── 1. is_active_paid 컬럼 생성 ──
# end_date가 비어 있음(해지 이력 없음) & 체험판이 아님 → 현재 활성 유료
subscriptions["is_active_paid"] = (
    subscriptions["end_date"].isna() & (~subscriptions["is_trial"].astype(bool))
)

print(f"현재 활성 유료 구독: {subscriptions['is_active_paid'].sum():,}건 "
      f"/ 전체 {len(subscriptions):,}건 "
      f"({subscriptions['is_active_paid'].mean()*100:.1f}%)\n")

# ── 2. plan_tier별 집계 ──
plan_metrics = subscriptions.groupby("plan_tier").agg(
    total_subs=("plan_tier", "size"),
    churned_subs=("churn_flag", "sum"),
    active_paid_subs=("is_active_paid", "sum"),
)
plan_metrics["churn_rate"] = plan_metrics["churned_subs"] / plan_metrics["total_subs"]

# ── 3. 활성 유료 구독만으로 요금제별 MRR 합계 ──
active_mrr = (subscriptions[subscriptions["is_active_paid"]]
              .groupby("plan_tier")["mrr_amount"]
              .sum())
plan_metrics["active_mrr"] = active_mrr          # 활성 구독이 없는 요금제는 NaN
plan_metrics["active_mrr"] = plan_metrics["active_mrr"].fillna(0)

plan_metrics = plan_metrics.sort_values("active_mrr", ascending=False)

# ── 4. 이탈률은 백분율, MRR은 천 단위 구분 기호로 출력 ──
display_df = pd.DataFrame({
    "전체 구독 수":      plan_metrics["total_subs"].map("{:,}".format),
    "이탈 구독 수":      plan_metrics["churned_subs"].astype(int).map("{:,}".format),
    "이탈률":            plan_metrics["churn_rate"].map("{:.1%}".format),
    "활성 유료 구독 수": plan_metrics["active_paid_subs"].astype(int).map("{:,}".format),
    "활성 MRR":          plan_metrics["active_mrr"].map("{:,.0f}".format),
})
print("===== 요금제별 지표 =====")
print(display_df.to_string())
print(f"\n전체 활성 유료 MRR 합계: {plan_metrics['active_mrr'].sum():,.0f}")

# ── 5. MRR 최대 / 이탈률 최대 요금제 ──
top_mrr_plan   = plan_metrics["active_mrr"].idxmax()
top_churn_plan = plan_metrics["churn_rate"].idxmax()

print(f"\n▶ MRR 최대 요금제  : {top_mrr_plan} "
      f"({plan_metrics.loc[top_mrr_plan, 'active_mrr']:,.0f}, "
      f"전체의 {plan_metrics.loc[top_mrr_plan,'active_mrr']/plan_metrics['active_mrr'].sum()*100:.1f}%)")
print(f"▶ 이탈률 최대 요금제: {top_churn_plan} "
      f"({plan_metrics.loc[top_churn_plan, 'churn_rate']:.1%}, "
      f"이탈 {int(plan_metrics.loc[top_churn_plan,'churned_subs']):,}건)")

# ── 6. 우선 점검 대상 판단용 보조 지표 ──
# 이탈률 × 활성 MRR = 해당 요금제에서 위험에 노출된 매출 규모
plan_metrics["mrr_at_risk"] = plan_metrics["churn_rate"] * plan_metrics["active_mrr"]
priority = plan_metrics.sort_values("mrr_at_risk", ascending=False)

print("\n===== 위험 노출 매출 기준 우선순위 =====")
print(pd.DataFrame({
    "이탈률":      priority["churn_rate"].map("{:.1%}".format),
    "활성 MRR":    priority["active_mrr"].map("{:,.0f}".format),
    "위험 MRR":    priority["mrr_at_risk"].map("{:,.0f}".format),
}).to_string())
print(f"\n▶ 우선 점검 권고: {priority.index[0]}")

현재 활성 유료 구독: 3,814건 / 전체 5,000건 (76.3%)

===== 요금제별 지표 =====
           전체 구독 수 이탈 구독 수    이탈률 활성 유료 구독 수     활성 MRR
plan_tier                                              
Enterprise   1,723     172  10.0%      1,304  7,546,876
Pro          1,675     162   9.7%      1,282  1,924,818
Basic        1,602     152   9.5%      1,228    687,914

전체 활성 유료 MRR 합계: 10,159,608

▶ MRR 최대 요금제  : Enterprise (7,546,876, 전체의 74.3%)
▶ 이탈률 최대 요금제: Enterprise (10.0%, 이탈 172건)

===== 위험 노출 매출 기준 우선순위 =====
              이탈률     활성 MRR   위험 MRR
plan_tier                            
Enterprise  10.0%  7,546,876  753,374
Pro          9.7%  1,924,818  186,162
Basic        9.5%    687,914   65,270

▶ 우선 점검 권고: Enterprise


### 필수 2 답변 작성란

- **Q1.** 
-> 전체 이탈률은 서비스 전체의 현상을 보여주고, 요금제별 이탈률은 어느 요금제를 먼저 점검해야하는 지를 판단한다.
-> 다음 회의가 열린다면 담당 팀이 개선 대상을 명확하게 설정하여 개선점을 진행할 수 있다.

- **Q2.** 
-> Enterprise이고 현재 유료 구독료는 $7,546,876이다.

- **Q3.**
-> Enterprise이고 이탈률은 9.98%이다.

- **Q4.**
-> Enterprise를 우선적으로 점검한다.
-> 현재 MRR 기여도가 가장 크면서 이탈률이 가장 높아
-> 이 그룹의 이탈률을 줄이는 것이 다음달 매출의 방어 효과가 가장 크다.

---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 유입 경로별 고객사 이탈률 비교하기

#### 문제 설명

Ravenstack 마케팅팀은 고객사가 유입된 경로에 따라 이탈 수준이 다른지 확인하려고 합니다. 유입 경로별 고객사 수와 이탈률을 비교하여 우선 점검할 유입 경로를 찾으세요.

> 이 과제는 필수 2에서 수행한 그룹별 지표 집계와 해석을 새로운 컬럼에 동일하게 적용하는 문제입니다.

#### 요구사항

1. `accounts`를 `referral_source`별로 그룹화하세요.
2. 다음 값을 집계하여 `source_metrics`를 만드세요.
   - 전체 고객사 수
   - 이탈 고객사 수
   - 고객사 이탈률
3. 고객사 이탈률이 높은 순서로 정렬하세요.
4. 이탈률은 소수점 둘째 자리의 백분율로 출력하세요.
5. 이탈률이 가장 높은 유입 경로를 찾으세요.
6. 해당 유입 경로를 우선 점검 대상으로 정하고, 확인할 개선 방향을 한 가지 제안하세요.
7. 유입 경로별 이탈률을 좋은 지표의 세 조건으로 평가하세요.
   - 행동 가능성
   - 비교 가능성
   - 이해 용이성

#### 해석 질문

**Q1.** 고객사 이탈률이 가장 높은 유입 경로는 무엇인가요?  
**Q2.** 유입 경로별 이탈률은 마케팅팀의 행동으로 어떻게 연결할 수 있나요?  
**Q3.** 유입 경로별 이탈률은 좋은 지표의 세 조건을 충족하나요?
**Q4.** KPI·보조 지표·허무 지표는 어떻게 다른가요? 고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시하세요. 허무 지표는 어떤 맥락에서 성과 판단에 도움이 되지 않는지도 설명하세요.

#### 제출 결과

- `source_metrics` 집계 코드와 결과
- 우선 점검할 유입 경로
- 개선 방향
- 좋은 지표의 조건 평가
- Q1~Q4 답변

In [4]:
import pandas as pd
import numpy as np

# ── 0. 이탈 컬럼 확인 (KeyError 방지) ──
print("accounts 컬럼:", list(accounts.columns))

CHURN_COL = next(
    (c for c in ["is_churned", "churned", "churn_flag", "is_churn"]
     if c in accounts.columns), None
)
if CHURN_COL is None:
    # 상태 문자열만 있는 경우 파생
    CHURN_COL = "is_churned"
    accounts[CHURN_COL] = (accounts["status"].astype(str).str.lower()
                           .isin(["churned", "cancelled", "canceled", "inactive"]))
accounts["_churned"] = pd.to_numeric(accounts[CHURN_COL], errors="coerce").fillna(0).astype(int)
print(f"사용할 이탈 컬럼: {CHURN_COL}\n")

# ── 1~2. referral_source별 그룹화 및 집계 ──
source_metrics = accounts.groupby("referral_source", dropna=False).agg(
    total_accounts=("account_id", "nunique"),
    churned_accounts=("_churned", "sum"),
)
source_metrics["churn_rate"] = (
    source_metrics["churned_accounts"] / source_metrics["total_accounts"]
)

# ── 3. 이탈률 내림차순 정렬 ──
source_metrics = source_metrics.sort_values("churn_rate", ascending=False)

# ── 4. 이탈률을 소수점 둘째 자리 백분율로 출력 ──
print("===== 유입 경로별 고객사 이탈률 =====")
print(pd.DataFrame({
    "전체 고객사 수": source_metrics["total_accounts"].map("{:,}".format),
    "이탈 고객사 수": source_metrics["churned_accounts"].astype(int).map("{:,}".format),
    "이탈률":        source_metrics["churn_rate"].map("{:.2%}".format),
}).to_string())

overall = accounts["_churned"].mean()
print(f"\n전체 평균 이탈률: {overall:.2%}")

# ── 5. 이탈률이 가장 높은 유입 경로 ──
worst = source_metrics["churn_rate"].idxmax()
w = source_metrics.loc[worst]
print(f"\n▶ 이탈률 최고 유입 경로: {worst}")
print(f"   이탈률 {w['churn_rate']:.2%} (전체 평균 대비 {w['churn_rate'] - overall:+.2%}p)")
print(f"   이탈 {int(w['churned_accounts']):,}개 / 전체 {int(w['total_accounts']):,}개")

# ── 6. 해석 보조: 표본 크기와 이탈 규모 ──
MIN_N = 30
small = source_metrics[source_metrics["total_accounts"] < MIN_N]
if len(small):
    print(f"\n     표본 {MIN_N}개 미만이라 비율이 불안정한 경로: {list(small.index)}")
    reliable = source_metrics[source_metrics["total_accounts"] >= MIN_N]
    if len(reliable):
        print(f"   표본 충분한 경로 중 최고: {reliable['churn_rate'].idxmax()} "
              f"({reliable['churn_rate'].max():.2%})")

print("\n===== 이탈 고객사 수 기준 (영향 규모) =====")
by_count = source_metrics.sort_values("churned_accounts", ascending=False)
print(pd.DataFrame({
    "이탈 고객사 수": by_count["churned_accounts"].astype(int).map("{:,}".format),
    "이탈률":        by_count["churn_rate"].map("{:.2%}".format),
}).to_string())

accounts 컬럼: ['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']
사용할 이탈 컬럼: churn_flag

===== 유입 경로별 고객사 이탈률 =====
                전체 고객사 수 이탈 고객사 수     이탈률
referral_source                          
event                 96       29  30.21%
other                103       25  24.27%
ads                   98       23  23.47%
organic              114       20  17.54%
partner               89       13  14.61%

전체 평균 이탈률: 22.00%

▶ 이탈률 최고 유입 경로: event
   이탈률 30.21% (전체 평균 대비 +8.21%p)
   이탈 29개 / 전체 96개

===== 이탈 고객사 수 기준 (영향 규모) =====
                이탈 고객사 수     이탈률
referral_source                 
event                 29  30.21%
other                 25  24.27%
ads                   23  23.47%
organic               20  17.54%
partner               13  14.61%


In [7]:
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
accounts['signup_year'] = accounts['signup_date'].dt.year
accounts['그룹'] = accounts['referral_source'].eq('event').map({True: 'event', False: '그 외'})

for col in ['signup_year', 'industry', 'country']:
    print(f"\n===== [{col}] event vs 그 외 구성 비율 (%) =====")
    print((pd.crosstab(accounts[col], accounts['그룹'], normalize='columns') * 100).round(1))
    print(f"\n===== [{col}]별 이탈률 (%) =====")
    print(accounts.groupby(col)['churn_flag'].mean().mul(100).round(1).sort_values(ascending=False))


===== [signup_year] event vs 그 외 구성 비율 (%) =====
그룹           event   그 외
signup_year             
2023          45.8  45.3
2024          54.2  54.7

===== [signup_year]별 이탈률 (%) =====
signup_year
2023    25.6
2024    19.0
Name: churn_flag, dtype: float64

===== [industry] event vs 그 외 구성 비율 (%) =====
그룹             event   그 외
industry                  
Cybersecurity   22.9  19.3
DevTools        24.0  22.3
EdTech           9.4  17.3
FinTech         27.1  21.3
HealthTech      16.7  19.8

===== [industry]별 이탈률 (%) =====
industry
DevTools         31.0
FinTech          22.3
HealthTech       21.9
EdTech           16.5
Cybersecurity    16.0
Name: churn_flag, dtype: float64

===== [country] event vs 그 외 구성 비율 (%) =====
그룹       event   그 외
country             
AU        10.4   5.4
CA         4.2   4.7
DE         0.0   6.2
FR         5.2   4.2
IN        10.4   9.7
UK        14.6  10.9
US        55.2  58.9

===== [country]별 이탈률 (%) =====
country
DE    32.0
US    23.4
FR    22.7
IN    20.4
UK 

### 과제 1 답변 작성란

- **Q1.** 고객사 이탈률이 가장 높은 유입 경로는 무엇인가요?
-> event. 30.21%로 고객 이탈률이 가장 높음.

- **Q2.** 유입 경로별 이탈률은 마케팅팀의 행동으로 어떻게 연결할 수 있나요?
-> 이탈률이 높은 event는 추가 분석 결과 체험판 여부, 요금제, 가입 시기, 업종, 국가 등 고객 특성으로 설명되지 않았다. 따라서 이벤트 이후의 경험을 점검 대상으로 삼고 이벤트 가입 고객에게 별도 온보딩 연락을 하는 등 후속 관리를 강화한다.
-> 이탈률이 낮은 partner와 organic은 유입된 고객이 잘 남아 있으므로 제휴를 늘리거나 검색 같은 자연 유입에 투자를 확대한다. 
-> 중간 수준인 ads는 추이를 지켜보고, other는 어떤 경로인지 알 수 없어 무엇을 고쳐야 할지 판단하기 어려우므로 유입 경로를 더 잘게 나눠 기록하도록 분류를 개선한다.
-> 이후 같은 지표로 경로별 이탈률을 다시 비교해 조치의 효과를 확인한다.

- **Q3.** 유입 경로별 이탈률은 좋은 지표의 세 조건을 충족하나요?
-> 유입 경로별 이탈률은 경로마다 다른 조치를 정할 수 있고, 비율이라 규모가 다른 경로끼리 비교되며, "10곳 중 3곳 이탈"처럼 바로 이해돼 세 조건을 대체로 충족한다.

- **Q4.** KPI·보조 지표·허무 지표는 어떻게 다른가요? 고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시하세요. 허무 지표는 어떤 맥락에서 성과 판단에 도움이 되지 않는지도 설명하세요.
-> KPI : 전체 구독 이탈률, 선정 이유는 이탈이 줄었는지를 바로 보여줌
-> 보조지표 : 현재 유료 구독 MRR / 유입 경로별 이탈률, 선정 이유는 MRR은 이탈이 매출에 준 영향을, 경로별 이탈률은 어디서 이탈이 생기는지를 보여줌
-> 허무지표: 누적 가입 고객사 , 선정 이유는 이탈한 고객사도 포함되어 있어 숫자가 늘기만 한다. 이탈이 늘어나는 상황에서도 성과가 좋아 보이게 만든다.

---

## 실습 마무리

아래 질문에 답하세요.

1. 이번 실습에서 해결하려고 한 비즈니스 문제는 무엇인가요?
-> 유료 구독을 안정적으로 유지하면서 반복 매출을 늘리기 위한 핵심 지표를 관리하고
-> 어느 대상을 우선적으로 개선해야 하는 지 확인했다.

2. 현재 목표를 직접 보여주는 KPI로 어떤 지표를 선택했나요?
-> 현재 활성화된 유료 구독자들이 만드는 매출인 MRR을 선택했다.

3. KPI의 변화를 이해하기 위해 어떤 보조 지표를 확인했나요?
-> 전체 구독 이탈률과 요금제별 이탈률을 확인했다.
-> 이탈률은 현재 또는 미래의 매출 감소로 이어지기 때문이다.

4. 허무 지표 후보를 핵심 성과로 사용할 때 어떤 문제가 생길 수 있나요?
-> 누적 가입 고객사를 핵심 성과로 사용한다면 현재 구독을 하지 않거나, 활동하지 않아도
-> 실제 성과를 과대평가 할 수 있다.

5. 어떤 분석 결과를 근거로 개선 대상을 정했나요?
-> 요금제별로 MRR과 이탈률을 분석하여 Enterprise를 우선 점검 대상으로 정했다.
